## Example 1: Basic Markdown parsing and rendering

In [ ]:
import mistune

# Example 1: Basic Markdown parsing and rendering
print("mistune basic Markdown parsing and rendering")

try:
    print("Example 1 - Basic Markdown parsing and rendering")
    
    # Sample Markdown content with various elements
    markdown_content = """
# Main Title

## Section Header

This is a paragraph with **bold text** and *italic text*.

Here's another paragraph with a [link to example](https://example.com).

### Lists

#### Unordered List:
- First item
- Second item with `inline code`
- Third item

#### Ordered List:
1. First numbered item
2. Second numbered item
3. Third numbered item

### Code Blocks

Here's a Python code block:

```python
def hello_world():
    print("Hello, World!")
    return "success"
```

And an inline code example: `print("Hello")`

### Blockquotes

> This is a blockquote with some important information.
> It can span multiple lines.

### Tables

| Name | Age | City |
|------|-----|------|
| John | 30  | NYC  |
| Jane | 25  | LA   |

---

That's the end of the sample content.
"""
    
    # Create markdown renderer. create_markdown() enables no plugin by default, and tables are a plugin,
    # so the Tables check below needs it named (without it the table renders as a <p>).
    markdown = mistune.create_markdown(plugins=["table"])
    
    # Parse markdown to HTML
    html_output = markdown(markdown_content)
    
    print(f"Markdown Input Length: {len(markdown_content)} characters")
    print(f"HTML Output Length: {len(html_output)} characters")
    print(f"Conversion successful: {html_output is not None}")
    
    # Display first few lines of HTML output
    html_lines = html_output.split('\n')
    print(f"\nFirst 15 lines of HTML output:")
    for i, line in enumerate(html_lines[:15], 1):
        print(f"  {i:2d}: {line}")
    
    # Verify key conversions
    conversions_verified = {
        'Headers': '<h1>' in html_output and '<h2>' in html_output,
        'Bold text': '<strong>bold text</strong>' in html_output,
        'Italic text': '<em>italic text</em>' in html_output,
        'Links': 'href="https://example.com"' in html_output,
        'Lists': '<ul>' in html_output and '<ol>' in html_output,
        'Code blocks': '<pre><code' in html_output,
        # The HTML renderer escapes quotes inside inline code.
        'Inline code': '<code>print(&quot;Hello&quot;)</code>' in html_output,
        'Blockquotes': '<blockquote>' in html_output,
        'Tables': '<table>' in html_output and '<th>' in html_output,
        'Horizontal rule': '<hr>' in html_output or '<hr />' in html_output
    }
    
    print(f"\nConversion Verification:")
    all_verified = True
    for element, verified in conversions_verified.items():
        status = "✓" if verified else "✗"
        print(f"  {status} {element}: {'Converted' if verified else 'Not found'}")
        all_verified &= verified
    
    print(f"\nOverall conversion success: {all_verified}")
    assert all(conversions_verified.values()), (
        f"conversions missing from the HTML: {[name for name, ok in conversions_verified.items() if not ok]}"
    )
    
    # Test different markdown variants
    print(f"\nTesting markdown variants:")
    
    test_cases = [
        ("# Simple header", "Simple header conversion"),
        ("**Bold** and *italic*", "Emphasis conversion"),
        ("`inline code`", "Inline code conversion"),
        ("- List item", "List conversion"),
        ("[link](http://example.com)", "Link conversion"),
        ("> Blockquote", "Blockquote conversion"),
        ("```\ncode block\n```", "Code block conversion")
    ]
    
    for markdown_test, description in test_cases:
        html_result = markdown(markdown_test)
        success = len(html_result) > 0 and html_result != markdown_test
        print(f"  {description}: {'✓ Success' if success else '✗ Failed'}")
        assert success, f"{description}: mistune returned {html_result!r}"
    
    # Check mistune version and features
    print(f"\nMistune Package Info:")
    if hasattr(mistune, '__version__'):
        print(f"  Version: {mistune.__version__}")
    
    print(f"  Available functions: {[attr for attr in dir(mistune) if not attr.startswith('_') and callable(getattr(mistune, attr))][:5]}")
    
    # Test empty and edge cases
    print(f"\nEdge case testing:")
    edge_cases = [
        ("", "Empty string"),
        ("   ", "Whitespace only"), 
        ("Plain text without markdown", "Plain text"),
        ("# ", "Header without text"),
        ("**", "Incomplete bold"),
        ("[]", "Empty link"),
        ("```\n```", "Empty code block")
    ]
    
    edge_success = 0
    for test_input, description in edge_cases:
        try:
            result = markdown(test_input)
            is_string = isinstance(result, str)
            print(f"  {description}: {'✓ Handled' if is_string else '✗ Failed'}")
            assert is_string, f"{description}: mistune returned a {type(result).__name__}, not a str"
            if is_string:
                edge_success += 1
        except Exception as e:
            print(f"  {description}: ✗ Error - {str(e)[:30]}...")
            raise
    
    print(f"  Edge case success rate: {edge_success}/{len(edge_cases)}")
    assert edge_success == len(edge_cases), edge_success
    
    print("mistune example 1 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"mistune example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: Custom renderers and advanced parsing

In [ ]:
import mistune
from mistune.renderers.html import HTMLRenderer

# Example 2: Custom renderers and advanced parsing
print("mistune custom renderers and advanced parsing")

try:
    print("Example 2 - Custom renderers and advanced parsing")
    
    # Create custom HTML renderer
    class CustomHTMLRenderer(HTMLRenderer):
        """Custom HTML renderer with specialized formatting"""
        
        def heading(self, text, level, **attrs):
            # Add custom classes to headings
            return f'<h{level} class="custom-heading level-{level}">{text}</h{level}>\n'
        
        def paragraph(self, text):
            # Add custom paragraph styling
            return f'<p class="custom-paragraph">{text}</p>\n'
        
        def link(self, text, url, title=None):
            # Add custom link attributes
            title_attr = f' title="{title}"' if title else ''
            return f'<a href="{url}" class="custom-link" target="_blank"{title_attr}>{text}</a>'
        
        def code_span(self, text):
            # Custom inline code formatting
            return f'<code class="inline-code">{text}</code>'
        
        # mistune 3's HTMLRenderer calls block_code and block_quote for fenced code and blockquotes.
        def block_code(self, code, info=None):
            # Custom code block with language support
            language = info or 'text'
            return f'<div class="code-block"><pre class="language-{language}"><code>{code}</code></pre></div>\n'
        
        def list_item(self, text):
            # Custom list item formatting
            return f'<li class="custom-list-item">{text}</li>\n'
        
        def table(self, text):
            # Custom table with wrapper
            return f'<div class="table-wrapper"><table class="custom-table">{text}</table></div>\n'
        
        def block_quote(self, text):
            # Enhanced blockquote
            return f'<blockquote class="custom-quote"><div class="quote-content">{text}</div></blockquote>\n'
    
    # Test markdown with custom renderer
    print("Testing custom HTML renderer:")
    
    test_markdown = """
# Custom Renderer Demo

## Features Showcase

This is a paragraph with **bold text**, *italic text*, and `inline code`.

Here's a [custom link](https://example.com "Link Title") with title.

### Code Example

```python
def greet(name):
    return f"Hello, {name}!"

result = greet("World")
print(result)
```

### Lists

- Item one with `code`
- Item two with [link](http://example.com)
- Item three

### Table

| Feature | Standard | Custom |
|---------|----------|---------|
| Headers | Basic | Styled |
| Links   | Plain | Enhanced |
| Code    | Simple | Highlighted |

### Quote

> This is a custom blockquote that demonstrates
> enhanced styling and formatting capabilities.

Regular paragraph to show the difference.
"""
    
    # Create custom markdown processor
    custom_renderer = CustomHTMLRenderer()
    # table() renders only when the table plugin parses the table; create_markdown() enables no plugin by default.
    custom_markdown = mistune.create_markdown(renderer=custom_renderer, plugins=["table"])
    
    # Process with custom renderer
    custom_html = custom_markdown(test_markdown)
    
    print(f"Custom HTML length: {len(custom_html)} characters")
    
    # Compare with default renderer
    default_markdown = mistune.create_markdown()
    default_html = default_markdown(test_markdown)
    
    print(f"Default HTML length: {len(default_html)} characters")
    
    # Show custom renderer features
    print(f"\nCustom renderer features detected:")
    custom_features = {
        'Custom heading classes': 'class="custom-heading' in custom_html,
        'Enhanced paragraphs': 'class="custom-paragraph"' in custom_html,
        'Styled links': 'class="custom-link"' in custom_html and 'target="_blank"' in custom_html,
        'Code block wrappers': 'class="code-block"' in custom_html,
        'Language detection': 'language-python' in custom_html,
        'Custom list items': 'class="custom-list-item"' in custom_html,
        'Table wrappers': 'class="table-wrapper"' in custom_html,
        'Enhanced blockquotes': 'class="custom-quote"' in custom_html
    }
    
    for feature, found in custom_features.items():
        status = "✓" if found else "○"
        print(f"  {status} {feature}: {'Found' if found else 'Not detected'}")
    
    assert all(custom_features.values()), (
        f"custom renderer hooks not reached: {[name for name, ok in custom_features.items() if not ok]}"
    )

    # Show first part of custom HTML
    print(f"\nSample custom HTML output (first 500 chars):")
    sample_custom = custom_html[:500].replace('\n', '\\n')
    print(f"  '{sample_custom}...'")
    
    # Advanced markdown features
    print(f"\n" + "="*40)
    print("Advanced markdown features:")
    
    # Test GitHub-flavored markdown features
    advanced_markdown = """
# Advanced Features Test

## Strikethrough
~~This text is crossed out~~

## Task Lists  
- [x] Completed task
- [ ] Pending task
- [x] Another completed task

## Tables with Alignment
| Left | Center | Right |
|:-----|:------:|------:|
| L1   |   C1   |    R1 |
| L2   |   C2   |    R2 |

## Fenced Code with Language
```javascript
function fibonacci(n) {
    if (n <= 1) return n;
    return fibonacci(n-1) + fibonacci(n-2);
}
```

## Autolinks
Visit https://www.example.com for more info.

## Line Breaks  
First line  
Second line (two spaces at end of first)

Third line after blank line.
"""
    
    # Create markdown with the strikethrough, task-list and table plugins
    plugins = ['strikethrough', 'task_lists', 'table']
    advanced_md = mistune.create_markdown(plugins=plugins)
    advanced_html = advanced_md(advanced_markdown)
    
    print(f"Advanced features HTML length: {len(advanced_html)} characters")
    
    # Check for advanced features
    advanced_features = {
        'Strikethrough': '<del>' in advanced_html or '<s>' in advanced_html,
        'Task lists': 'type="checkbox"' in advanced_html,
        'Table alignment': 'text-align:' in advanced_html or 'style=' in advanced_html,
        'Code language': 'class=' in advanced_html or 'language-' in advanced_html,
        'Line breaks': '<br' in advanced_html
    }
    
    print(f"Advanced features support:")
    for feature, supported in advanced_features.items():
        status = "✓" if supported else "○"
        print(f"  {status} {feature}: {'Supported' if supported else 'Basic only'}")
    
    assert all(advanced_features.values()), (
        f"plugin output missing: {[name for name, ok in advanced_features.items() if not ok]}"
    )

    # Performance comparison
    print(f"\n" + "="*40)
    print("Renderer performance comparison:")
    
    import time
    
    # Performance test content
    perf_content = test_markdown * 5  # Repeat content
    
    # Test default renderer
    start = time.time()
    for _ in range(50):
        default_markdown(perf_content)
    default_time = time.time() - start
    
    # Test custom renderer  
    start = time.time()
    for _ in range(50):
        custom_markdown(perf_content)
    custom_time = time.time() - start
    
    print(f"Default renderer (50 iterations): {default_time:.4f}s")
    print(f"Custom renderer (50 iterations): {custom_time:.4f}s")
    print(f"Performance ratio: {custom_time/default_time:.2f}x")
    
    # Analyze HTML structure differences
    print(f"\nHTML structure analysis:")
    
    default_lines = default_html.split('\n')
    custom_lines = custom_html.split('\n')
    
    print(f"  Default HTML: {len(default_lines)} lines")
    print(f"  Custom HTML: {len(custom_lines)} lines")
    
    # Count HTML tags
    import re
    default_tags = re.findall(r'<[^>]+>', default_html)
    custom_tags = re.findall(r'<[^>]+>', custom_html)
    
    print(f"  Default tags: {len(default_tags)}")
    print(f"  Custom tags: {len(custom_tags)}")
    
    # Show renderer customization capabilities
    print(f"\nRenderer customization capabilities:")
    customization_features = [
        "Custom HTML classes: Add styling hooks for CSS frameworks",
        "Element wrapping: Wrap elements in additional containers",
        "Attribute modification: Add/modify HTML attributes dynamically",
        "Content transformation: Process text content before rendering",
        "Plugin integration: Support for markdown extensions",
        "Performance control: Optimize rendering for specific use cases"
    ]
    
    for feature in customization_features:
        print(f"  • {feature}")
    
    print("mistune example 2 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"mistune example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: Plugin system and extensibility

In [ ]:
import mistune
from mistune import create_markdown

# Example 3: Plugin system and extensibility
print("mistune plugin system and extensibility")

try:
    print("Example 3 - Plugin system and extensibility")
    
    # Test built-in plugins
    print("Testing built-in plugins:")
    
    available_plugins = [
        'strikethrough',
        'mark', 
        'insert',
        'superscript',
        'subscript',
        'footnotes',
        'table',
        'task_lists',
        'def_list',
        'abbr',
        'url',
        'spoiler'
    ]
    
    # Test which plugins are available
    working_plugins = []
    plugin_test_content = {
        'strikethrough': '~~strikethrough text~~',
        'mark': '==highlighted text==',
        'insert': '^^inserted text^^',
        'superscript': 'x^2^',
        'subscript': 'H~2~O',
        'footnotes': 'Text with footnote[^1]\n\n[^1]: Footnote content',
        'table': '| A | B |\n|---|---|\n| 1 | 2 |',
        'task_lists': '- [x] Done\n- [ ] Todo',
        'def_list': 'Term\n: Definition',
        # A definition alone renders to nothing; the abbreviation has to be used to show the plugin working.
        'abbr': 'The HTML spec\n\n*[HTML]: HyperText Markup Language',
        'url': 'https://www.example.com',
        'spoiler': 'Text with >!hidden words!< inline'
    }
    
    # The HTML each plugin adds; rendering the same text without the plugin must not produce it.
    plugin_markers = {
        'strikethrough': '<del>',
        'mark': '<mark>',
        'insert': '<ins>',
        'superscript': '<sup>',
        'subscript': '<sub>',
        'footnotes': 'footnote-ref',
        'table': '<table>',
        'task_lists': 'type="checkbox"',
        'def_list': '<dl>',
        'abbr': '<abbr',
        'url': '<a href=',
        'spoiler': 'class="spoiler"'
    }
    plain_md = create_markdown()
    
    for plugin in available_plugins:
        try:
            # Create markdown with single plugin
            md = create_markdown(plugins=[plugin])
            test_content = plugin_test_content[plugin]
            result = md(test_content)
            marker = plugin_markers[plugin]
            
            # The plugin worked if its marker appears with the plugin and not without it
            processed = marker in result and marker not in plain_md(test_content)
            if processed:
                working_plugins.append(plugin)
                print(f"  ✓ {plugin}: Working")
            else:
                print(f"  ○ {plugin}: No effect detected")
                
        except Exception as e:
            print(f"  ✗ {plugin}: Error - {str(e)[:40]}...")
            raise
    
    print(f"\nWorking plugins: {working_plugins}")
    assert working_plugins == available_plugins, (
        f"plugins with no effect: {[p for p in available_plugins if p not in working_plugins]}"
    )
    
    # Create comprehensive markdown with multiple plugins
    if working_plugins:
        print(f"\nTesting combined plugins:")
        
        combined_content = """
# Plugin Demonstration

## Text Formatting Extensions

Regular **bold**, *italic*, and ~~strikethrough~~ text.

==Highlighted text== and ^^inserted text^^.

Mathematical expressions: E=mc^2^ and chemical formulas: H~2~O.

## Lists and Tasks

- [x] Completed task
- [ ] Pending task  
- [x] Another done task

Regular list:
- Item 1
- Item 2

## Tables

| Feature | Status | Notes |
|---------|--------|-------|
| Tables | ✓ | Working |
| Tasks | ✓ | Checkboxes |
| Strike | ✓ | Text decoration |

## Links and URLs

Auto-link: https://www.example.com

Regular [link](https://example.com) format.

## Definition Lists

HTML
: HyperText Markup Language

CSS  
: Cascading Style Sheets

## Spoilers and Special Content

Inline spoiler: >!This is spoiler content that should be hidden!<

## Footnotes

This has a footnote[^1] and another[^note].

[^1]: First footnote content
[^note]: Named footnote content

## Abbreviations

The HTML specification defines web standards.

*[HTML]: HyperText Markup Language
"""
        
        # Test with different plugin combinations
        plugin_combinations = [
            ['strikethrough', 'table'],
            ['mark', 'insert', 'superscript', 'subscript'],
            ['task_lists', 'footnotes'],
            working_plugins[:5],  # First 5 working plugins
            working_plugins  # All working plugins
        ]
        
        for i, plugin_combo in enumerate(plugin_combinations, 1):
            # Filter to only working plugins
            valid_plugins = [p for p in plugin_combo if p in working_plugins]
            
            if valid_plugins:
                try:
                    combo_md = create_markdown(plugins=valid_plugins)
                    combo_result = combo_md(combined_content)
                    
                    print(f"  Combination {i} ({', '.join(valid_plugins)}): {len(combo_result)} chars")
                    
                    # Check for specific plugin features
                    features_found = []
                    if 'strikethrough' in valid_plugins and '<del>' in combo_result:
                        features_found.append('strikethrough')
                    if 'table' in valid_plugins and '<table>' in combo_result:
                        features_found.append('table')
                    if 'task_lists' in valid_plugins and 'type="checkbox"' in combo_result:
                        features_found.append('task_lists')
                    if 'footnotes' in valid_plugins and 'footnote' in combo_result.lower():
                        features_found.append('footnotes')
                    
                    print(f"    Features detected: {features_found}")
                    
                except Exception as e:
                    print(f"    Error with combination {i}: {str(e)[:40]}...")
                    raise
    
    # The last combination enables every working plugin, so all four detected features must appear.
    assert features_found == ['strikethrough', 'table', 'task_lists', 'footnotes'], features_found

    # Custom plugin development example
    print(f"\n" + "="*40)
    print("Custom plugin development:")
    
    # Custom {{highlight}} syntax through a renderer subclass that rewrites text nodes
    try:
        print("Testing custom text rendering:")
        
        class HighlightRenderer(mistune.renderers.html.HTMLRenderer):
            def text(self, text):
                # Simple post-processing approach
                import re
                highlighted = re.sub(r'\{\{([^}]+)\}\}', r'<span class="highlight">\1</span>', text)
                return highlighted
        
        highlight_renderer = HighlightRenderer()
        highlight_md = create_markdown(renderer=highlight_renderer)
        
        custom_test = """
# Custom Plugin Test

Regular text and {{highlighted text}} in the same paragraph.

Another paragraph with {{multiple}} {{highlights}} here.

- List item with {{highlighted}} content
- Regular list item

{{This entire sentence is highlighted}}.
"""
        
        custom_result = highlight_md(custom_test)
        
        highlight_count = custom_result.count('<span class="highlight">')
        print(f"  Custom highlighting: {highlight_count} highlights found")
        assert highlight_count == 5, highlight_count
        print(f"  Result length: {len(custom_result)} characters")
        
        if highlight_count > 0:
            print(f"  ✓ Custom plugin concept working")
        else:
            print(f"  ○ Custom plugin needs different approach")
        
    except Exception as e:
        print(f"  Custom plugin error: {str(e)[:50]}...")
        raise
    
    # Plugin architecture analysis
    print(f"\n" + "="*40)
    print("Plugin architecture analysis:")
    
    # Examine markdown instance structure
    default_md = create_markdown()
    
    print(f"Markdown instance analysis:")
    
    # Plugin-related attributes of a mistune 3 Markdown instance; plugins are applied with use()
    plugin_attrs = [
        'use',
        'inline',
        'block',
        'renderer',
        'parse',
        'render_state'
    ]
    
    for attr in plugin_attrs:
        if hasattr(default_md, attr):
            attr_value = getattr(default_md, attr)
            attr_type = type(attr_value).__name__
            print(f"  ✓ {attr}: {attr_type}")
            
            # Show methods for renderers
            if attr == 'renderer' and hasattr(attr_value, '__dict__'):
                methods = [m for m in dir(attr_value) if not m.startswith('_') and callable(getattr(attr_value, m))]
                print(f"    Methods: {methods[:8]}{'...' if len(methods) > 8 else ''}")
        else:
            print(f"  ○ {attr}: Not available")
    assert all(hasattr(default_md, attr) for attr in plugin_attrs), [a for a in plugin_attrs if not hasattr(default_md, a)]
    
    # Security considerations
    print(f"\nPlugin security considerations:")
    
    security_test_content = """
# Security Test

<script>alert('xss')</script>

<img src=x onerror=alert('xss')>

[evil link](javascript:alert('xss'))

<iframe src="javascript:alert('xss')"></iframe>
"""
    
    # Test with different configurations
    configs = [
        ("Default", {}),
        ("Escape HTML", {'escape': True}),
        ("Hard wrap", {'hard_wrap': True})
    ]
    
    for config_name, config in configs:
        try:
            test_md = create_markdown(**config)
            result = test_md(security_test_content)
            
            # Check for dangerous content
            dangerous_patterns = ['<script>', 'javascript:', 'onerror=', '<iframe']
            security_issues = sum(1 for pattern in dangerous_patterns if pattern in result.lower())
            
            print(f"  {config_name}: {security_issues} potential security issues")
            # create_markdown() escapes raw HTML by default, so no live <script> or <iframe> tag may survive;
            # the `javascript:` / `onerror=` hits above are inside that escaped text.
            assert '<script>' not in result.lower() and '<iframe' not in result.lower(), (config_name, result)
            
        except Exception as e:
            print(f"  {config_name}: Error - {str(e)[:40]}...")
            raise
    
    # Show plugin ecosystem benefits
    print(f"\nPlugin ecosystem benefits:")
    ecosystem_benefits = [
        "Extensibility: Add custom markdown syntax and rendering",
        "Modularity: Enable only needed features for performance",
        "Compatibility: Support for GitHub-flavored markdown and more",
        "Customization: Tailor output format for specific applications",
        "Security: Control which features are enabled for user content",
        "Performance: Choose optimal feature set for your use case"
    ]
    
    for benefit in ecosystem_benefits:
        print(f"  • {benefit}")
    
    print("mistune example 3 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"mistune example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: Performance optimization and real-world usage

In [ ]:
import mistune
import time

# Example 4: Performance optimization and real-world usage
print("mistune performance optimization and real-world usage")

try:
    print("Example 4 - Performance optimization and real-world usage")
    
    # Generate performance test content
    def generate_test_content(size='medium'):
        """Generate markdown content of various sizes for testing"""
        
        if size == 'small':
            multiplier = 1
        elif size == 'medium':
            multiplier = 5
        elif size == 'large':
            multiplier = 20
        else:
            multiplier = 50  # extra_large
        
        content = f"""
# Performance Test Document {size.title()}

## Overview
This document tests mistune performance with various content types and sizes.

## Text Content
""" + "\n\n".join([f"This is paragraph number {i+1} with **bold text**, *italic text*, and `inline code`. " +
                   f"It contains [a link](https://example.com/page/{i}) and various formatting." 
                   for i in range(10 * multiplier)]) + """

## Code Blocks

Here are some code examples:

""" + "\n\n".join([f"""```python
def function_{i}():
    \"\"\"Example function number {i}\"\"\"
    result = []
    for j in range(10):
        result.append(f"Item {{j}} from function {i}")
    return result

# Call the function
data_{i} = function_{i}()
print(f"Generated {{len(data_{i})}} items")
```""" for i in range(2 * multiplier)]) + f"""

## Lists and Tables

### Unordered Lists
""" + "\n\n".join([f"""
- Main item {i+1}
  - Sub-item {i+1}.1 with `code`
  - Sub-item {i+1}.2 with [link](https://example.com)
  - Sub-item {i+1}.3 with **bold** text
""" for i in range(3 * multiplier)]) + f"""

### Tables

""" + "\n\n".join([f"""
| Column A | Column B | Column C | Column D |
|----------|----------|----------|----------|
| Row {i} A | Row {i} B | Row {i} C | Row {i} D |
| Value {i*2} | Value {i*2+1} | Value {i*2+2} | Value {i*2+3} |
""" for i in range(2 * multiplier)]) + f"""

## Blockquotes and Special Content

""" + "\n\n".join([f"""
> This is blockquote number {i+1}. It contains important information
> about performance testing and markdown rendering capabilities.
> It may span multiple lines for testing purposes.
""" for i in range(2 * multiplier)]) + """

## Final Section

This concludes the performance test document.
"""
        
        return content
    
    # Performance benchmark
    print("Performance benchmarking:")
    
    # Test different content sizes
    sizes = ['small', 'medium', 'large']
    test_contents = {size: generate_test_content(size) for size in sizes}
    
    # Create different markdown configurations
    configurations = {
        'default': mistune.create_markdown(),
        'minimal': mistune.create_markdown(plugins=[]),
        'full_featured': mistune.create_markdown(plugins=['strikethrough', 'table', 'task_lists']),
    }
    
    performance_results = {}
    
    for config_name, markdown_instance in configurations.items():
        performance_results[config_name] = {}
        
        print(f"\n{config_name.title()} Configuration:")
        
        for size in sizes:
            content = test_contents[size]
            
            # Warm-up
            for _ in range(3):
                markdown_instance(content[:1000])
            
            # Benchmark
            iterations = 50 if size == 'small' else (20 if size == 'medium' else 5)
            
            start_time = time.time()
            for _ in range(iterations):
                result = markdown_instance(content)
            end_time = time.time()
            
            total_time = end_time - start_time
            avg_time = total_time / iterations
            chars_per_sec = len(content) / avg_time
            
            performance_results[config_name][size] = {
                'total_time': total_time,
                'avg_time': avg_time,
                'chars_per_sec': chars_per_sec,
                'content_length': len(content),
                'iterations': iterations
            }
            
            print(f"  {size:6s}: {avg_time*1000:6.2f}ms avg ({chars_per_sec:8.0f} chars/sec) - {len(content):,} chars")
    
    # Performance comparison
    print(f"\n" + "="*50)
    print("Performance comparison:")
    
    for size in sizes:
        print(f"\n{size.title()} content ({test_contents[size][:50]}...):")
        baseline = performance_results['default'][size]['avg_time']
        
        for config_name in configurations.keys():
            result = performance_results[config_name][size]
            ratio = result['avg_time'] / baseline
            
            print(f"  {config_name:12s}: {result['avg_time']*1000:6.2f}ms ({ratio:5.2f}x baseline)")
    
    # Real-world usage patterns
    print(f"\n" + "="*50)
    print("Real-world usage patterns:")
    
    # Documentation generation
    class DocumentationGenerator:
        """Generate documentation from markdown files"""
        
        def __init__(self, custom_renderer=None):
            plugins = ['table', 'strikethrough', 'task_lists']
            # `renderer=None` is NOT "use the default" in mistune 3 — it is the AST mode, and the
            # parser then returns a token LIST instead of HTML. Every downstream helper here does
            # string work on the result (`re.findall` over `<h1>` tags), so the default must be the
            # html renderer, with None meaning "caller did not override it".
            # escape=False: documentation is trusted input, and create_markdown() escapes raw HTML by default,
            # which would render the metadata comment as visible text.
            self.markdown = mistune.create_markdown(
                plugins=plugins,
                escape=False,
                renderer=custom_renderer if custom_renderer is not None else 'html'
            )
        
        def process_documentation_file(self, content, metadata=None):
            """Process a documentation markdown file"""
            # Add metadata processing
            if metadata:
                content = f"<!-- Metadata: {metadata} -->\n\n{content}"
            
            # Process markdown
            html_content = self.markdown(content)
            
            # Post-processing for documentation
            html_content = self.add_table_of_contents(html_content)
            html_content = self.add_code_highlighting(html_content)
            
            return html_content
        
        def add_table_of_contents(self, html):
            """Add table of contents based on headers"""
            import re
            
            # Find all headers
            headers = re.findall(r'<h([1-6])[^>]*>([^<]+)</h[1-6]>', html)
            
            if len(headers) > 1:
                toc_items = []
                for level, text in headers:
                    indent = '  ' * (int(level) - 1)
                    anchor = text.lower().replace(' ', '-').replace('/', '')
                    toc_items.append(f'{indent}- [{text}](#{anchor})')
                
                toc_html = f"""
<div class="table-of-contents">
<h2>Table of Contents</h2>
<ul>
{''.join(f'<li>{item}</li>' for item in toc_items)}
</ul>
</div>

"""
                return toc_html + html
            
            return html
        
        def add_code_highlighting(self, html):
            """Add syntax highlighting classes to code blocks"""
            import re
            
            # Add highlighting classes
            highlighted = re.sub(
                r'<pre><code class="language-(\w+)">',
                r'<pre class="highlight"><code class="language-\1">',
                html
            )
            
            return highlighted
    
    # Test documentation generation
    doc_generator = DocumentationGenerator()
    
    doc_content = """
# API Documentation

## Overview
This API provides access to user management functionality.

## Authentication
All requests require authentication via API key.

### Example Request
```bash
curl -H "Authorization: Bearer YOUR_API_KEY" \\
     https://api.example.com/users
```

## Endpoints

### GET /users
Retrieve user list.

| Parameter | Type | Required | Description |
|-----------|------|----------|-------------|
| page      | int  | No       | Page number |
| limit     | int  | No       | Results per page |

### POST /users
Create a new user.

```json
{
  "name": "John Doe",
  "email": "john@example.com"
}
```

## Error Handling
API returns standard HTTP status codes.

~~Deprecated endpoints are marked with strikethrough.~~

## Tasks
- [x] Implement authentication
- [ ] Add rate limiting
- [x] Write documentation
"""
    
    processed_doc = doc_generator.process_documentation_file(
        doc_content, 
        metadata={'version': '1.0', 'author': 'API Team'}
    )
    
    print(f"Documentation generation:")
    print(f"  Input length: {len(doc_content):,} characters")
    print(f"  Output length: {len(processed_doc):,} characters")
    print(f"  Processing overhead: {(len(processed_doc)/len(doc_content)-1)*100:.1f}%")
    
    # Check for documentation features
    doc_features = {
        'Table of contents': 'table-of-contents' in processed_doc,
        'Metadata comments': '<!-- Metadata:' in processed_doc,
        'Code highlighting': 'class="highlight"' in processed_doc,
        'Tables processed': '<table>' in processed_doc,
        'Task lists processed': 'type="checkbox"' in processed_doc,
        'Strikethrough processed': '<del>' in processed_doc
    }
    
    for feature, found in doc_features.items():
        status = "✓" if found else "○"
        print(f"  {status} {feature}: {'Working' if found else 'Not detected'}")
    
    assert all(doc_features.values()), (
        f"documentation features missing: {[name for name, ok in doc_features.items() if not ok]}"
    )

    # Content management system usage
    print(f"\nContent Management System usage:")
    
    class CMSMarkdownProcessor:
        """Process markdown content for a CMS"""
        
        def __init__(self):
            # CMS-optimized configuration
            self.markdown = mistune.create_markdown(
                plugins=['table', 'strikethrough'],
                escape=False  # CMS handles escaping
            )
        
        def process_blog_post(self, markdown_content, allow_html=False):
            """Process blog post content"""
            if not allow_html:
                # Strip HTML for security
                import re
                markdown_content = re.sub(r'<[^>]+>', '', markdown_content)
            
            html_content = self.markdown(markdown_content)
            
            # Add CMS-specific features
            html_content = self.add_read_more_breaks(html_content)
            html_content = self.process_image_galleries(html_content)
            
            return html_content
        
        def add_read_more_breaks(self, html):
            """Add read more functionality"""
            # Simple read more after first 2 paragraphs
            import re
            paragraphs = html.split('</p>')
            if len(paragraphs) > 2:
                excerpt = '</p>'.join(paragraphs[:2]) + '</p>'
                full_content = '</p>'.join(paragraphs[2:])
                return f'{excerpt}\n<div class="read-more-break"></div>\n{full_content}'
            return html
        
        def process_image_galleries(self, html):
            """Convert consecutive images to galleries"""
            # Simple gallery detection (consecutive img tags)
            import re
            
            # Find consecutive images
            gallery_pattern = r'(<img[^>]+>)\s*(<img[^>]+>)+'
            
            def gallery_replacement(match):
                images = re.findall(r'<img[^>]+>', match.group(0))
                gallery_html = '<div class="image-gallery">'
                for img in images:
                    gallery_html += f'<div class="gallery-item">{img}</div>'
                gallery_html += '</div>'
                return gallery_html
            
            return re.sub(gallery_pattern, gallery_replacement, html)
    
    # Test CMS processing
    cms_processor = CMSMarkdownProcessor()
    
    blog_content = """
# My Blog Post

This is the introduction paragraph that will appear in the excerpt.

This is the second paragraph that's also part of the excerpt.

This content will appear after the "read more" break.

## Gallery Section

![Image 1](image1.jpg)
![Image 2](image2.jpg)
![Image 3](image3.jpg)

## More Content

Regular paragraph after the gallery.
"""
    
    processed_blog = cms_processor.process_blog_post(blog_content)
    
    print(f"Blog post processing:")
    print(f"  Read more breaks: {'read-more-break' in processed_blog}")
    print(f"  Image galleries: {'image-gallery' in processed_blog}")
    assert 'read-more-break' in processed_blog and 'image-gallery' in processed_blog, processed_blog[:300]
    print(f"  Content length: {len(processed_blog):,} characters")
    
    # Show real-world optimization benefits
    print(f"\nReal-world optimization benefits:")
    optimization_benefits = [
        "Configuration tuning: Choose optimal plugin set for your use case",
        "Content caching: Cache processed HTML for frequently accessed content",
        "Batch processing: Process multiple documents efficiently",
        "Custom renderers: Optimize output for your specific HTML structure",
        "Security controls: Configure escaping and allowed features appropriately",
        "Performance monitoring: Track processing times for large content volumes"
    ]
    
    for benefit in optimization_benefits:
        print(f"  • {benefit}")
    
    print("mistune example 4 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"mistune example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: Advanced mistune features and production workflow

In [ ]:
import mistune

# Example 5: mistune advanced features and production workflow  
print("mistune advanced features and production workflow")

try:
    print("Example 5 - Advanced mistune features and production workflow")
    
    # Demonstrate package structure and capabilities
    print(f"mistune Package Overview:")
    print(f"  Module type: {type(mistune).__name__}")
    print(f"  Package location: {mistune.__file__ if hasattr(mistune, '__file__') else 'Built-in'}")
    
    # Package version and metadata
    if hasattr(mistune, '__version__'):
        print(f"  Version: {mistune.__version__}")
    
    # Key functions and classes
    key_components = {
        'create_markdown': 'Main factory function for markdown processors',
        'HTMLRenderer': 'HTML output renderer class',
        'Markdown': 'Core markdown processing class'
    }
    
    print(f"  Key Components:")
    for component, description in key_components.items():
        try:
            if hasattr(mistune, component):
                comp = getattr(mistune, component)
                comp_type = 'function' if callable(comp) else 'class'
                print(f"    ✓ {component} ({comp_type}): {description}")
            else:
                print(f"    ○ {component}: Not directly accessible")
        except AttributeError:
            print(f"    ✗ {component}: Not available")
    
    # Production workflow demonstration
    print(f"\nProduction Workflow Pattern:")
    
    # Step 1: Configuration management
    print(f"  1. Configuration Management:")
    
    # Different configuration profiles for different environments
    config_profiles = {
        'development': {
            'plugins': ['strikethrough', 'table', 'task_lists'],
            'escape': False,
            'description': 'Full features for development'
        },
        'production': {
            'plugins': ['table'],
            'escape': True,
            'description': 'Minimal secure features for production'
        },
        'blog': {
            'plugins': ['strikethrough', 'table'],
            'escape': False,
            'description': 'Blog-optimized feature set'
        },
        'documentation': {
            'plugins': ['table', 'task_lists'],
            'escape': False,
            'description': 'Documentation with tables and task tracking'
        }
    }
    
    markdown_instances = {}
    
    for profile_name, config in config_profiles.items():
        try:
            plugins = config.get('plugins', [])
            markdown_instance = mistune.create_markdown(escape=config['escape'], plugins=plugins)
            markdown_instances[profile_name] = markdown_instance
            
            print(f"     ✓ {profile_name}: {config['description']}")
            print(f"       Plugins: {plugins}, escape={config['escape']}")
            
        except Exception as e:
            print(f"     ✗ {profile_name}: Error - {str(e)[:40]}...")
            raise
    
    # escape=True renders raw HTML as text; escape=False passes it through
    raw_html = "<script>alert('xss')</script>\n\nSome **text**."
    assert '<script>' not in markdown_instances['production'](raw_html), markdown_instances['production'](raw_html)
    assert '&lt;script&gt;' in markdown_instances['production'](raw_html)
    assert '<script>' in markdown_instances['development'](raw_html), markdown_instances['development'](raw_html)
    print("     ✓ production profile escapes raw HTML; development passes it through")
    
    # Step 2: Content validation and preprocessing
    print(f"  2. Content Validation:")
    
    class MarkdownValidator:
        """Validate markdown content before processing"""
        
        @staticmethod
        def validate_content(content):
            """Basic content validation"""
            issues = []
            
            # Check for basic issues
            if not content or not content.strip():
                issues.append("Content is empty")
            
            if len(content) > 1000000:  # 1MB limit
                issues.append("Content exceeds size limit")
            
            # Check for suspicious patterns
            suspicious_patterns = [
                (r'<script[^>]*>', 'Potential script injection'),
                (r'javascript:', 'JavaScript URL detected'),
                (r'data:text/html', 'Data URL with HTML'),
                (r'<iframe[^>]*>', 'Embedded iframe detected')
            ]
            
            import re
            for pattern, description in suspicious_patterns:
                if re.search(pattern, content, re.IGNORECASE):
                    issues.append(f"Security: {description}")
            
            return issues
        
        @staticmethod
        def preprocess_content(content):
            """Preprocess content for consistency"""
            # Normalize line endings
            content = content.replace('\r\n', '\n').replace('\r', '\n')
            
            # Remove excessive blank lines
            import re
            content = re.sub(r'\n{3,}', '\n\n', content)
            
            # Trim whitespace
            content = content.strip()
            
            return content
    
    # Test content validation
    validator = MarkdownValidator()
    
    test_contents = [
        "# Valid Content\n\nThis is safe markdown.",
        "<script>alert('xss')</script>\n# Dangerous Content",
        "",  # Empty content
        "# Normal\n\n\n\n\n\nToo many blank lines",
        "[Link](javascript:alert('evil'))"
    ]
    
    for i, content in enumerate(test_contents, 1):
        issues = validator.validate_content(content)
        clean_content = validator.preprocess_content(content) if not issues else content
        
        sample = content[:30].replace('\n', '\\n') if content else '(empty)'
        print(f"     Content {i} ({sample}...): {len(issues)} issues")
        
        for issue in issues[:2]:  # Show first 2 issues
            print(f"       - {issue}")
    
    # Step 3: Processing with error handling
    print(f"  3. Processing with Error Handling:")
    
    class ProductionMarkdownProcessor:
        """Production-ready markdown processor with error handling"""
        
        def __init__(self, profile='production'):
            self.profile = profile
            self.validator = MarkdownValidator()
            
            # Get configuration
            config = config_profiles.get(profile, config_profiles['production'])
            plugins = config.get('plugins', [])
            
            try:
                self.markdown = mistune.create_markdown(escape=config['escape'], plugins=plugins)
                self.ready = True
            except Exception as e:
                self.markdown = mistune.create_markdown()  # Fallback to basic
                self.ready = False
                self.error = str(e)
        
        def process_content(self, content, validate=True):
            """Process markdown content with full error handling"""
            result = {
                'success': False,
                'html': '',
                'errors': [],
                'warnings': [],
                'metadata': {}
            }
            
            try:
                # Validation
                if validate:
                    issues = self.validator.validate_content(content)
                    if issues:
                        result['errors'].extend(issues)
                        return result
                
                # Preprocessing
                content = self.validator.preprocess_content(content)
                
                # Processing
                html_output = self.markdown(content)
                
                # Post-processing validation
                if not html_output:
                    result['warnings'].append("Empty output generated")
                
                if len(html_output) > len(content) * 10:
                    result['warnings'].append("Output significantly larger than input")
                
                # Success
                result['success'] = True
                result['html'] = html_output
                result['metadata'] = {
                    'input_length': len(content),
                    'output_length': len(html_output),
                    'profile': self.profile,
                    'processor_ready': self.ready
                }
                
            except Exception as e:
                result['errors'].append(f"Processing error: {str(e)}")
            
            return result
    
    # Test production processor
    for profile in ['production', 'blog', 'documentation']:
        if profile in config_profiles:
            processor = ProductionMarkdownProcessor(profile)
            
            test_content = """
# Production Test

This is a **test** of the *production* processor.

| Feature | Status |
|---------|--------|
| Tables  | ✓      |
| Security| ✓      |

- [x] Basic formatting
- [ ] Advanced features

~~Deprecated content~~
"""
            
            result = processor.process_content(test_content)
            
            print(f"     {profile.title()} profile:")
            print(f"       Success: {result['success']}")
            print(f"       Errors: {len(result['errors'])}")
            print(f"       Warnings: {len(result['warnings'])}")
            assert result['success'] and not result['errors'], (profile, result['errors'])
            
            if result['metadata']:
                meta = result['metadata']
                print(f"       Output: {meta['output_length']} chars from {meta['input_length']} chars")
    
    # Step 4: Performance monitoring
    print(f"  4. Performance Monitoring:")
    
    class PerformanceMonitor:
        """Monitor markdown processing performance"""
        
        def __init__(self):
            self.metrics = {}
        
        def time_processing(self, markdown_instance, content, name="default"):
            """Time a markdown processing operation"""
            import time
            
            start_time = time.time()
            result = markdown_instance(content)
            end_time = time.time()
            
            processing_time = end_time - start_time
            
            if name not in self.metrics:
                self.metrics[name] = []
            
            self.metrics[name].append({
                'time': processing_time,
                'input_length': len(content),
                'output_length': len(result),
                'chars_per_second': len(content) / processing_time if processing_time > 0 else float('inf')
            })
            
            return result, processing_time
        
        def get_summary(self):
            """Get performance summary"""
            summary = {}
            
            for name, measurements in self.metrics.items():
                if measurements:
                    times = [m['time'] for m in measurements]
                    speeds = [m['chars_per_second'] for m in measurements if m['chars_per_second'] != float('inf')]
                    
                    summary[name] = {
                        'count': len(measurements),
                        'avg_time': sum(times) / len(times),
                        'min_time': min(times),
                        'max_time': max(times),
                        'avg_speed': sum(speeds) / len(speeds) if speeds else 0
                    }
            
            return summary
    
    # Test performance monitoring
    monitor = PerformanceMonitor()
    
    # Sample content of different sizes
    small_content = "# Small\n\nJust a **quick** test."
    medium_content = "# Medium\n\n" + "This is a paragraph with *formatting*.\n\n" * 20
    large_content = "# Large\n\n" + "This is a much longer paragraph with **bold** and *italic* text and `code`.\n\n" * 100
    
    test_cases = [
        ('small', small_content),
        ('medium', medium_content), 
        ('large', large_content)
    ]
    
    # Monitor different processors
    for profile_name, md_instance in markdown_instances.items():
        for size_name, content in test_cases:
            test_name = f"{profile_name}_{size_name}"
            
            # Run multiple times for better average
            for _ in range(5):
                try:
                    monitor.time_processing(md_instance, content, test_name)
                except Exception as e:
                    print(f"       Error in {test_name}: {str(e)[:30]}...")
                    raise
    
    # Show performance summary
    summary = monitor.get_summary()
    print(f"     Performance Summary:")
    
    for test_name, metrics in sorted(summary.items()):
        print(f"       {test_name:20s}: {metrics['avg_time']*1000:6.2f}ms avg ({metrics['avg_speed']:8.0f} chars/sec)")
    
    # Step 5: Production best practices
    print(f"  5. Production Best Practices:")
    
    best_practices = [
        "Input validation: Always validate content before processing",
        "Error handling: Implement comprehensive error handling and logging",
        "Security configuration: Use appropriate escaping and plugin settings",
        "Performance monitoring: Track processing times and throughput",
        "Configuration profiles: Different settings for different environments",
        "Content caching: Cache processed output for frequently accessed content",
        "Resource limits: Set limits on content size and processing time",
        "Fallback mechanisms: Graceful degradation when processing fails"
    ]
    
    for i, practice in enumerate(best_practices, 1):
        print(f"     {i}. {practice}")
    
    # Package ecosystem integration
    print(f"\nEcosystem Integration:")
    integrations = [
        "Web frameworks: Flask, Django, FastAPI integration",
        "Static site generators: Jekyll, Hugo, Pelican compatibility",
        "Documentation tools: Sphinx, MkDocs, GitBook workflows",
        "Content management: WordPress, Drupal, custom CMS systems",
        "API services: RESTful APIs for markdown processing",
        "Real-time editing: Live preview and collaborative editing"
    ]
    
    for integration in integrations:
        print(f"  • {integration}")
    
    # Production deployment considerations
    print(f"\nProduction Deployment Considerations:")
    deployment_considerations = [
        "Memory usage: Monitor memory consumption with large documents",
        "Concurrent processing: Handle multiple requests efficiently",
        "Security policies: Content Security Policy and XSS prevention",
        "Error logging: Comprehensive logging for debugging and monitoring",
        "Performance tuning: Optimize for your specific content patterns",
        "Scalability: Horizontal scaling and load balancing strategies"
    ]
    
    for consideration in deployment_considerations:
        print(f"  • {consideration}")
    
    print("mistune example 5 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"mistune example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")